In [2]:
from datasets import load_dataset, disable_progress_bars
from collections import Counter
disable_progress_bars()

dataset = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset")

data = dataset["train"]
# cancel_order, change_order, change_shipping_address, track_order, contact_human_agent, get_refund

intents = ["cancel_order", "change_order", "change_shipping_address", "track_order", "contact_human_agent", "get_refund"]
examples_by_intent = {intent: [] for intent in intents}

selected_data = data.filter(lambda row: row["intent"] in intents)

for row in data:
    intent = row["intent"]

    if intent in examples_by_intent and len(examples_by_intent[intent]) < 3:
        examples_by_intent[intent].append(row)

    if all(len(examples_by_intent[intent]) == 3 for intent in intents):
        break

for intent in intents:
    print(f"\n--- {intent} ---")
    for row in examples_by_intent[intent]:
        print(f"Instruction: {row['instruction']}, Intent: {row['intent']}")
from collections import Counter

counts = Counter(data["intent"])


c:\Users\alkur\Desktop\Complex Adaptive Systems\Own Projects\support-ticket-assistant\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



--- cancel_order ---
Instruction: question about cancelling order {{Order Number}}, Intent: cancel_order
Instruction: i have a question about cancelling oorder {{Order Number}}, Intent: cancel_order
Instruction: i need help cancelling puchase {{Order Number}}, Intent: cancel_order

--- change_order ---
Instruction: I try to change several bloody items of order {{Order Number}}, Intent: change_order
Instruction: modifying purchase {{Order Number}}, Intent: change_order
Instruction: want help adding an item to order {{Order Number}}, Intent: change_order

--- change_shipping_address ---
Instruction: give me information about a delivery address modification, Intent: change_shipping_address
Instruction: need to update my address, Intent: change_shipping_address
Instruction: I entered a wrong address, update it, Intent: change_shipping_address

--- track_order ---
Instruction: needx help to check the ETA of purchase {{Order Number}}, Intent: track_order
Instruction: ETA of purchase {{Order

In [3]:
for intent in intents:
    instruction_counts = Counter(
        row["instruction"]
        for row in data
        if row["intent"] == intent
    )

    duplicate_texts = sum(
        1 for count in instruction_counts.values() if count > 1
    )

    extra_duplicate_rows = sum(
        count - 1 for count in instruction_counts.values() if count > 1
    )

    print(
        f"{intent}: {duplicate_texts} duplicate instruction texts, "
        f"{extra_duplicate_rows} extra duplicate rows"
    )

cancel_order: 224 duplicate instruction texts, 505 extra duplicate rows
change_order: 65 duplicate instruction texts, 127 extra duplicate rows
change_shipping_address: 0 duplicate instruction texts, 0 extra duplicate rows
track_order: 111 duplicate instruction texts, 188 extra duplicate rows
contact_human_agent: 0 duplicate instruction texts, 0 extra duplicate rows
get_refund: 21 duplicate instruction texts, 79 extra duplicate rows


In [4]:
from collections import defaultdict

labels_by_instruction = defaultdict(set)

for row in data:
    if row["intent"] in intents:
        instruction = row.get("instruction")

        if instruction is not None and instruction.strip() != "":
            labels_by_instruction[instruction].add(row["intent"])

conflicts = {
    instruction: labels
    for instruction, labels in labels_by_instruction.items()
    if len(labels) > 1
}

print(f"Conflicting instruction texts: {len(conflicts)}")

for instruction, labels in list(conflicts.items())[:10]:
    print(f"{instruction!r} -> {sorted(labels)}")

Conflicting instruction texts: 0


In [12]:
import pandas as pd 
dataset = load_dataset(
    "bitext/Bitext-customer-support-llm-chatbot-training-dataset",
    revision="430d1a89bd93bd1fa23c16f29dd53e73f0087443",
)

dataset = dataset["train"]

intents = [
    "cancel_order",
    "change_order",
    "change_shipping_address",
    "track_order",
    "contact_human_agent",
    "get_refund",
]

selected_data = dataset.filter(lambda row: row["intent"] in intents)
selected_data = selected_data.add_column("row_id", list(range(len(selected_data))))
dataset_df = selected_data.to_pandas()

data_row_id = pd.read_csv(r"C:\Users\alkur\Desktop\Complex Adaptive Systems\Own Projects\support-ticket-assistant\split_ids.csv")

combined_data = pd.merge(dataset_df, data_row_id, on="row_id")

combined_data.shape
combined_data["split"].value_counts()

split
train          4171
validation      596
calibration     596
test            596
Name: count, dtype: int64

In [29]:
Counter(selected_data["intent"])
validation_data = combined_data[combined_data["split"] == "validation"]
validation_data["intent"].value_counts()

correct = (validation_data["intent"] == "contact_human_agent").sum() 
total = validation_data["intent"].value_counts().sum()
accuracy = correct / total
print(correct, total, accuracy)

training_data = combined_data[combined_data["split"] == "train"]
training_data["intent"].value_counts()

100 596 0.16778523489932887


intent
cancel_order               699
contact_human_agent        699
change_order               698
get_refund                 697
track_order                697
change_shipping_address    681
Name: count, dtype: int64

In [42]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report

X_train = training_data["instruction"]
y_train = training_data["intent"]
y_validation = validation_data["intent"]
X_validation = validation_data["instruction"]

pipe = Pipeline([
    ('Tfidf', TfidfVectorizer()),
    ('logistic', LogisticRegression())
])

pipe.fit(X_train, y_train)

y_pred = pipe.predict(X_validation)

accuracy = accuracy_score(y_validation,y_pred)
class_report = classification_report(y_validation, y_pred)
print(accuracy)
print(class_report)

0.9932885906040269
                         precision    recall  f1-score   support

           cancel_order       0.99      1.00      0.99        99
           change_order       0.97      1.00      0.99       100
change_shipping_address       1.00      1.00      1.00        98
    contact_human_agent       1.00      1.00      1.00       100
             get_refund       1.00      1.00      1.00       100
            track_order       1.00      0.96      0.98        99

               accuracy                           0.99       596
              macro avg       0.99      0.99      0.99       596
           weighted avg       0.99      0.99      0.99       596



In [44]:
wrong = y_validation.to_numpy() != y_pred

mistakes = validation_data.loc[wrong, ["instruction", "intent"]].copy()
mistakes["predicted_intent"] = y_pred[wrong]

mistakes

,instruction,intent,predicted_intent
5026,trackorder {{Order Number}},track_order,change_order
5302,trackorder {{Order Number}},track_order,change_order
5704,I need assistance trackng purchase {{Order Num...,track_order,cancel_order
5810,trackorder {{Order Number}},track_order,change_order


In [6]:
import joblib 
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.linear_model import LogisticRegression

pipe = Pipeline([
    ('Tfidf', TfidfVectorizer()),
    ('logistic', LogisticRegression())
])
joblib.dump(pipe, "artifacts/tfidf_baseline.joblib")

['artifacts/tfidf_baseline.joblib']

In [ ]:

test = joblib.load("artifacts/tfidf_baseline.joblib")
test.predict(X_validation) == y_pred

NameError: name 'X_validation' is not defined